# 🚀 LLM Gateway Explained — Build One With LiteLLM + LangChain

---

## 📺 What You'll Learn in This Video

In this hands-on tutorial, we'll cover:

1. **What is an LLM Gateway?** — The problem it solves
2. **Why do we need it?** — Real production pain points
3. **Core capabilities** — Routing, fallbacks, caching, observability, cost tracking
4. **Practical implementation** — Build one from scratch using `LiteLLM`
5. **Integration with LangChain** — Plug the gateway into your agentic apps
6. **Production patterns** — Logging, retries, multi-provider fallbacks

By the end, you'll have a **working LLM gateway** that routes between OpenAI, Anthropic, and Groq — with caching, fallbacks, and cost tracking built in. 🔥

---

## 🧠 Part 1: What is an LLM Gateway?

Think of an **LLM Gateway** as a **smart middleware layer** that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

```
                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
```

### Without a Gateway (The Pain 😩)

- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

### With a Gateway (The Joy 😎)

- **One unified API** for 100+ providers
- **Automatic fallbacks** if a provider fails
- **Centralized logging, cost tracking, rate limiting**
- **Swap models with a config change**, no code rewrite
- **Cache repeated queries** → save money

## ⚙️ Part 2: Installation & Setup

We'll use:
- **LiteLLM** → the most popular open-source LLM gateway (supports 100+ providers)
- **LangChain** → for building agentic workflows on top of the gateway
- **python-dotenv** → for managing API keys

In [24]:
from dotenv import load_dotenv
import warnings, logging, os, time, re

import litellm
from collections import Counter
from litellm.caching import Cache
from litellm import completion, completion_cost, Router
from IPython.display import Markdown, display

from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

litellm.suppress_debug_info = True

- **API Keys: OpenAI, Anthropic, Groq, Google**

In [2]:
# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")
print("Google key loaded:    ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")

OpenAI key loaded:     ❌
Anthropic key loaded:  ❌
Groq key loaded:       ✅
Google key loaded:     ✅


## 🎯 Part 3: The Simplest LiteLLM Example — Unified API

The biggest pain point: **every provider has a different SDK**.

LiteLLM gives you **one function** — `completion()` — that works with all of them. Look at how clean this is:

In [7]:
# Same code, different providers — just change the `model` string!

TEST_PROMPT = "What is full form of LLM? (Only Abbreviation, no explanation)"

# Call OpenAI model
response_openai = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": TEST_PROMPT}
    ]
)
print("🔵 OpenAI:    ", response_openai.choices[0].message.content)

# Call Google model
response_groq = completion(
    model="gemini/gemini-3.5-flash",
    messages=[
        {"role": "user", "content": TEST_PROMPT}
    ]
)
print("🟢 Google:    ", response_groq.choices[0].message.content)

🔵 OpenAI:     Large Language Model
🟢 Google:     Large Language Model


**🎉 Notice:** Same code, three different providers. This alone is huge — you can switch providers with a string change.

But a real LLM Gateway does much more. Let's build those features one by one. 👇

In [11]:
response_groq.choices[0].message.content

'Large Language Model'

In [17]:
# Test prompt for multiple providers
TEST_PROMPT = "Explain the RAG (Retrieval-Augmented Generation) in one short sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        response = completion(
            model=model,
            messages=[
                {"role": "user", "content": TEST_PROMPT}
            ]
        )
        display(Markdown(f"**{label}**: {response.choices[0].message.content}"))
    except Exception as e:
        display(Markdown(f"**{label}**: ❌ {type(e).__name__}"))

**🔵 OpenAI**: ❌ InternalServerError

**🟢 Groq**: RAG (Retrieval-Augmented Generation) is a natural language processing technique that combines a retriever model to fetch relevant information with a generator model to produce output based on the retrieved context.

**🟣 Anthropic**: ❌ BadRequestError

**🟡 Gemini**: Retrieval-Augmented Generation (RAG) is an AI technique that retrieves relevant information from external sources to help a language model generate more accurate and up-to-date answers.

## 🛡️ Part 4: Automatic Fallbacks — When OpenAI Goes Down

**Real story:** OpenAI had a 4-hour outage in November 2023. Apps that hard-coded `gpt-4` went completely dark.

With a gateway, if one provider fails, we **automatically fall back** to another. Production apps must have this.

In [ ]:
# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": TEST_PROMPT}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.3-70b-versatile"
    ]
)


display(Markdown(f"**Response:** {response.choices[0].message.content}"))
display(Markdown(f"**Model used:** {response.model}"))

16:37:57 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 3100, in async_completion
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\llms\custom

**Response:** RAG (Retrieval-Augmented Generation) is a natural language processing technique that combines retrieval of relevant information with generation of text to improve the accuracy and coherence of generated content.

**Model used:** llama-3.3-70b-versatile

If `OpenAI` is rate-limited or down (API not set), LiteLLM transparently retries with Claude, then Groq. Your app **never sees the failure**.

This is the #1 reason teams adopt an LLM Gateway.

In [21]:
# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # 👈 will fail intentionally
    messages=[{"role": "user", "content": TEST_PROMPT}],
    fallbacks=[
        "gpt-4o-mini",                              # 1st backup: real OpenAI model
        "groq/llama-3.3-70b-versatile"              # 2nd backup: Groq
    ]
)


display(Markdown(f"**Response:** {response.choices[0].message.content}"))
display(Markdown(f"**Model used:** {response.model}"))

16:40:01 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model openai/fake-nonexistent-model-9999: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Traceback (most recent call last):
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 904, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(  # type: ignore
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 386, in _get_openai_client
    _new_client: Union[OpenAI, AsyncOpenAI] = AsyncOpenAI(
                                              ^^^^^^^^^^^^
  File "c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\openai\_client.py", line 826, in __in

Task was destroyed but it is pending!
task: <Task pending name='Task-167' coro=<LoggingWorker._worker_loop() running at c:\GenAI-AgenticAI-MCP-Tutorials\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:111>>


**Response:** RAG (Retrieval-Augmented Generation) is a natural language processing technique that combines information retrieval with text generation to produce more accurate and informative outputs.

**Model used:** llama-3.3-70b-versatile

## 💰 Part 5: Cost Tracking — Know Where Your Money Goes

LiteLLM **automatically calculates the cost** of every call using its built-in pricing database. No more surprise bills.

In [42]:
# Call OpenAI model
response = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": TEST_PROMPT}],
)

display(Markdown(f"**Response:** {response.choices[0].message.content}"))
display(Markdown(f"**Model used:** {response.model}"))

# Get the exact USD cost of this single call
cost = litellm.completion_cost(completion_response=response, model="groq/llama-3.1-8b-instant")

display(Markdown(f"**💰 Cost of this call:** ${cost:.6f}"))
display(Markdown(f"**📝 Input Tokens:** {response.usage.prompt_tokens}"))
display(Markdown(f"**📝 Output Tokens:** {response.usage.completion_tokens}"))

**Response:** RAG (Retrieval-Augmented Generation) is a type of AI model that combines text retrieval and generation, where a generative model uses previously retrieved relevant information to produce output.

**Model used:** llama-3.1-8b-instant

**💰 Cost of this call:** $0.000006

**📝 Input Tokens:** 53

**📝 Output Tokens:** 38

## ⚡ Part 6: Caching — Don't Pay Twice for the Same Question

If 100 users ask *"What is RAG?"*, you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [43]:
# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [57]:
# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

# Define a prompt
PROMPT = "❄️  What is full form of LLM? (Only Abbreviation, no explanation)"


# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": PROMPT}],
    caching=True
)
t1 = time.time() - start
display(Markdown(f"**First call response:** {t1:.2f}s {r1.choices[0].message.content}"))

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": PROMPT}],
    caching=True
)
t2 = time.time() - start
display(Markdown(f"**⚡ Second call response:** {t2:.4f}s {r2.choices[0].message.content}"))

display(Markdown(f"\n🚀 **Speedup:** {t1/t2:.6f}x faster, and ZERO cost on the second call!"))


**First call response:** 0.09s Large Language Model

**⚡ Second call response:** 0.0013s Large Language Model


🚀 **Speedup:** 68.906823x faster, and ZERO cost on the second call!

## 🔀 Part 7: Smart Routing — The Right Model for the Right Job

**Why use one model for everything?**

- Coding tasks → Claude Sonnet
- Cheap summaries → GPT-4o-mini
- Super fast replies → Groq Llama
- Complex reasoning → Claude Opus

Use LiteLLM's **Router** to define routing rules:

In [58]:
model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                             
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",                  
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "groq/qwen/qwen3-32b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    }
]

In [64]:
router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

display(Markdown(f"**Fast response:** {fast_response.choices[0].message.content}"))
display(Markdown(f"**Model used:** {fast_response.model}"))

display(Markdown(f"**=============================================**"*3))

display(Markdown(f"**Code response:** {code_response.choices[0].message.content}"))
display(Markdown(f"**Model used:** {code_response.model}"))

**Fast response:** Artificial intelligence (AI) is revolutionizing the software industry in several ways:

1. **Automated coding**: AI-powered tools can generate code, reducing the need for human programmers and increasing development speed.
2. **Intelligent assistance**: AI-powered assistants can help developers with tasks such as debugging, testing, and optimizing code.
3. **Personalized user experiences**: AI-driven software can analyze user behavior and provide personalized recommendations, improving user engagement and satisfaction.
4. **Predictive maintenance**: AI-powered software can predict and prevent errors, reducing downtime and improving overall system reliability.
5. **New business models**: AI is enabling new business models, such as subscription-based services and pay-per-use models, which are changing the way software is sold and delivered.
6. **Improved security**: AI-powered software can detect and prevent cyber threats, improving the security of software systems.
7. **Enhanced analytics**: AI-driven software can analyze large datasets, providing insights and patterns that can inform business decisions.

Overall, AI is transforming the software industry by increasing efficiency, improving user experiences, and enabling new business models and revenue streams.

**Model used:** llama-3.3-70b-versatile

**=============================================****=============================================****=============================================**

**Code response:** Sure! Below is a concise, well‑documented Python function that reverses a string. I’ll also include a few alternative approaches so you can pick the one that best fits your style or performance needs.

```python
def reverse_string(s: str) -> str:
    """
    Return a new string that is the reverse of the input string `s`.

    Parameters
    ----------
    s : str
        The string to reverse.

    Returns
    -------
    str
        The reversed string.

    Examples
    --------
    >>> reverse_string("hello")
    'olleh'
    >>> reverse_string("")
    ''
    """
    # The most Pythonic way: slicing with a step of -1
    return s[::-1]
```

### How it works
- `s[::-1]` is a slice operation.
- The first colon (`:`) means “take the whole string”.
- The second colon (`:`) introduces the *step* argument.
- A step of `-1` tells Python to walk through the string backwards, producing a reversed copy.

### Alternative implementations

| Method | Description | When to use |
|--------|-------------|-------------|
| `''.join(reversed(s))` | Uses the built‑in `reversed` iterator and then joins the characters back together. | When you already have an iterator or want an explicit “reversal” operation. |
| `list(s).reverse(); ''.join(list(s))` | Converts to a list, reverses it in‑place, then joins. | Useful if you need to mutate a list of characters elsewhere. |
| Recursive function | Demonstrates recursion (inefficient for long strings). | Educational purposes only. |
| Loop & accumulator | Manually builds the reversed string with a `for` loop. | When you want full control (e.g., adding extra logic per character). |

#### Example of the alternative approaches

```python
# 1. Using reversed() + join
def reverse_string_via_reversed(s: str) -> str:
    return ''.join(reversed(s))

# 2. Using an explicit loop
def reverse_string_via_loop(s: str) -> str:
    rev = []
    for ch in s:
        rev.insert(0, ch)          # prepend each character
    return ''.join(rev)

# 3. Recursive version (not recommended for large strings)
def reverse_string_recursive(s: str) -> str:
    if len(s) <= 1:
        return s
    return reverse_string_recursive(s[1:]) + s[0]
```

### Quick test

```python
if __name__ == "__main__":
    test_strings = ["hello", "Python 🐍", "", "A"]
    for txt in test_strings:
        print(f"{txt!r} -> {reverse_string(txt)!r}")
```

**Output**

```
'hello' -> 'olleh'
'Python 🐍' -> '🐍 nohtyP'
'' -> ''
'A' -> 'A'
```

Feel free to pick the version you like best, or adapt the function for more complex use‑cases (e.g., handling Unicode normalization, preserving whitespace, etc.). Happy coding!

**Model used:** openai/gpt-oss-120b

**💡 Key insight:** Your app calls `"fast-cheap"` or `"smart-coding"` — abstract names. The router decides which provider to actually use. Tomorrow, you can swap Groq for a cheaper provider with **zero code changes**.

## 🔁 Part 8: Load Balancing Across Multiple API Keys

Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [70]:
# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "openai-oss-120b"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Summarize: AI is changing software. (Request {i+1})"}]
    )
    
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "Unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq-llama-70b           520 ms   AI is transforming the software ind
#2        groq-llama-70b           682 ms   AI is revolutionizing the software 
#3        groq-llama-70b           996 ms   Here's a summary of how AI is chang
#4        groq-llama-70b           512 ms   AI is transforming the software ind
#5        groq-llama-70b           691 ms   Here's a summary of how AI is chang
#6        groq-llama-70b           963 ms   Here's a summary of how AI is chang


### 🎯 Strategy 1: least-busy —
 The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [71]:
model_list = [
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/openai/gpt-oss-120b",
                            "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🔵 OpenAI"}
    },
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🟢 Groq"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 OpenAI
Request 2 → 🔵 OpenAI
Request 3 → 🔵 OpenAI
Request 4 → 🔵 OpenAI
Request 5 → 🔵 OpenAI
Request 6 → 🔵 OpenAI
Request 7 → 🔵 OpenAI
Request 8 → 🔵 OpenAI

🎯 Distribution:
   🔵 OpenAI: ████████ (8)


### 🎯 Strategy 2: latency-based-routing — 
The "Always Pick the Fastest" Pattern
The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins.

In [72]:
model_list = [
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/openai/gpt-oss-120b",
                            "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🔵 OpenAI"}
    },
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🟢 Groq"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🟢 Groq                             482 ms
#2    🔵 OpenAI                           321 ms
#3    🔵 OpenAI                           349 ms
#4    🔵 OpenAI                           372 ms
#5    🔵 OpenAI                           309 ms
#6    🔵 OpenAI                           279 ms
#7    🔵 OpenAI                           451 ms
#8    🔵 OpenAI                           353 ms
#9    🔵 OpenAI                           329 ms
#10   🔵 OpenAI                           327 ms


Expected behavior: First 2-3 requests will be exploratory (router doesn't have latency data yet), then it'll lock onto whichever deployment is consistently fastest — usually Groq, since it specializes in fast inference

### 🎯 Strategy 4: cost-based-routing — The "Always Cheapest" Pattern
The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

In [73]:
model_list = [
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/openai/gpt-oss-120b",
                            "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🔵 OpenAI"}
    },
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🟢 Groq"}
    },
    {
        "model_name": "chat",
        "litellm_params": {"model": "groq/llama-3.3-8b-versatile",
                            "api_key": os.getenv("GROQ_API_KEY")},
        "model_info": {"id": "🟡 Groq"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   # 👈 valid strategy
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🟢 Groq
Request 2 → 🟢 Groq
Request 3 → 🟢 Groq
Request 4 → 🔵 OpenAI
Request 5 → 🟢 Groq


## 📊 Part 9: Observability — Log Every Single Call

In production, you **must** log every LLM call: prompt, response, latency, cost, user_id, etc.

LiteLLM supports custom callbacks — here's a simple logger:

In [75]:
# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "What is RAG?",
    "input_tokens": 76,
    "output_tokens": 1881,
    "latency_sec": 4.33,
    "cost_usd": 0.00114,
    "user": "krish"
  },
  {
    "model": "openai/gpt-oss-120b",
    "prompt": "Explain transformers.",
    "input_tokens": 74,
    "output_tokens": 3072,
    "latency_sec": 6.51,
    "cost_usd": 0.0018543,
    "user": "student_42"
  }
]


Now you have a **per-user, per-call audit trail** — exactly what you need for chargebacks, debugging, and security reviews.

## 🔗 Part 10: Integrating the Gateway with LangChain

Here's where it really clicks for production GenAI apps:

**LangChain** for the orchestration (agents, chains, RAG) + **LiteLLM** as the unified LLM backend.

LangChain has a built-in `ChatLiteLLM` wrapper — drop it in like any other chat model.

In [77]:
# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
display(Markdown(f"**Answer:** {answer}"))

**Answer:** - **Unified API layer** that routes requests from applications to one or multiple large language models (LLMs), handling authentication, rate‑limiting, and versioning.  
- **Model‑agnostic orchestration** that can select, combine, or switch between different LLM providers (e.g., OpenAI, Anthropic, local models) based on cost, latency, or capability criteria.  
- **Safety & governance hub** that enforces content filters, logging, usage quotas, and compliance policies before responses are returned to the client.

**🎯 The magic:** swap `model="gpt-4o-mini"` → `"claude-3-5-sonnet-20241022"` → `"groq/llama-3.3-70b-versatile"` and the *entire chain* now runs on a different provider. Zero other changes.

## 🤖 Part 11: A Real Example — Multi-Provider LangChain Chain with Fallbacks

Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [78]:
# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

❌ Call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-x
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
{"answer":"1. **Unified Access & Management** – A single entry point to multiple LLM providers simplifies integration, version control, and monitoring, reducing development overhead and ensuring consistent security policies.\n2. **Scalability & Load Balancing** – The gateway can route requests across providers or model instances, automatically handling traffic spikes, fail‑over, and cost‑optimisation by selecting the most appropriate model for each task.\n3. **Enhanced Security & Governance** – Centralised authentication, data sanitisation, usage logging, and policy enforcement protect sensitive data and help organisations meet compliance requirements while auditing LLM usage."}


If Claude fails (rate limit, outage, etc.), the chain transparently retries with GPT, then Groq. **Your downstream code never knows.**

## 🧪 Part 13: A Mini End-to-End Demo — Smart Router for a Chatbot

Let's build a tiny **task-aware chatbot** that:

1. Decides what kind of question the user is asking (code, summary, general)
2. Routes to the right model accordingly
3. Falls back if the chosen model fails
4. Logs cost and latency

In [80]:
# A simple classifier
def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error


def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["groq/openai/gpt-oss-120b",                     "groq/openai/gpt-oss-20b",   "groq/llama-3.3-70b-versatile"],
        "summary": ["groq/qwen/qwen3-32b",                "groq/llama-3.3-70b-versatile"],
        "general": ["groq/llama-3.3-70b-versatile", "groq/qwen/qwen3-32b"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }


# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

❓ Q: Write a Python function to compute Fibonacci numbers.
🏷️  Task:    code
🤖 Model:    openai/gpt-oss-120b
⏱️  Latency: 2.33s
💰 Cost:    $0.000000
💬 Answer:  Here’s a compact, easy‑to‑read implementation of the Fibonacci sequence in Python.  
It includes a **fast, iterative version** (O(n) time, O(1) space) and a **memoized recursive version** (still O(n) ...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.
🏷️  Task:    summary
🤖 Model:    qwen/qwen3-32b
⏱️  Latency: 0.0s
💰 Cost:    n/a
💬 Answer:  

Attention mechanisms enhance model performance by dynamically emphasizing the most relevant input features, enabling more accurate and context-aware processing. They also facilitate handling long-ra...
❓ Q: Tell me a fun fact about elephants.
🏷️  Task:    general
🤖 Model:    llama-3.3-70b-versatile
⏱️  Latency: 0.0s
💰 Cost:    n/a
💬 Answer:  Here's a fun fact: Elephants have a highly developed sense of empathy and self-awareness. They are one of the few animals that ca